# House Price Prediction

A complete machine-learning project using `Housing.csv` to predict house prices.

**Workflow:** data loading → EDA → preprocessing → train/test split → Linear Regression & Random Forest → evaluation → prediction.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")


## 1. Load and inspect the dataset

In [ ]:
df = pd.read_csv("Housing.csv")
print("Dataset shape:", df.shape)
display(df.head())


In [ ]:
df.info()
display(df.describe(include="all").T)
print("Missing values:")
display(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())


## 2. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(9,5))
sns.histplot(df["price"], kde=True)
plt.title("Distribution of House Prices")
plt.xlabel("Price")
plt.show()


In [ ]:
plt.figure(figsize=(9,5))
sns.scatterplot(data=df, x="area", y="price")
plt.title("House Price vs Area")
plt.xlabel("Area")
plt.ylabel("Price")
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x="bedrooms", y="price")
plt.title("House Price by Bedrooms")
plt.show()


In [ ]:
plt.figure(figsize=(9,5))
sns.boxplot(data=df, x="furnishingstatus", y="price")
plt.title("House Price by Furnishing Status")
plt.show()


### Additional Data Visualizations

The following graphs give a deeper view of the relationships between house features and price.

In [ ]:
# Correlation heatmap for numerical features
plt.figure(figsize=(10, 7))
numeric_df = df.select_dtypes(include=np.number)
sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()


In [ ]:
# Average house price by number of bathrooms
avg_bathroom_price = df.groupby('bathrooms')['price'].mean().sort_index()
plt.figure(figsize=(8, 5))
sns.barplot(x=avg_bathroom_price.index, y=avg_bathroom_price.values)
plt.title('Average House Price by Number of Bathrooms')
plt.xlabel('Bathrooms')
plt.ylabel('Average Price')
plt.tight_layout()
plt.show()


In [ ]:
# Average price by number of stories
avg_story_price = df.groupby('stories')['price'].mean().sort_index()
plt.figure(figsize=(8, 5))
sns.barplot(x=avg_story_price.index, y=avg_story_price.values)
plt.title('Average House Price by Number of Stories')
plt.xlabel('Stories')
plt.ylabel('Average Price')
plt.tight_layout()
plt.show()


In [ ]:
# Price comparison for air conditioning
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='airconditioning', y='price')
plt.title('House Price by Air Conditioning')
plt.xlabel('Air Conditioning')
plt.ylabel('Price')
plt.tight_layout()
plt.show()


In [ ]:
# Price comparison by preferred area
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='prefarea', y='price')
plt.title('House Price by Preferred Area')
plt.xlabel('Preferred Area')
plt.ylabel('Price')
plt.tight_layout()
plt.show()


In [ ]:
# Distribution of house area
plt.figure(figsize=(9, 5))
sns.histplot(df['area'], bins=30, kde=True)
plt.title('Distribution of House Area')
plt.xlabel('Area')
plt.ylabel('Number of Houses')
plt.tight_layout()
plt.show()


## 3. Prepare features and target

In [ ]:
X = df.drop(columns=["price"])
y = df["price"]

categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_features = X.select_dtypes(exclude=["object", "category"]).columns.tolist()

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])


## 4. Train Linear Regression

In [ ]:
linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])
linear_model.fit(X_train, y_train)
linear_predictions = linear_model.predict(X_test)


## 5. Train Random Forest

In [ ]:
random_forest_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=300, random_state=42, n_jobs=-1
    ))
])
random_forest_model.fit(X_train, y_train)
rf_predictions = random_forest_model.predict(X_test)


## 6. Evaluate the models

In [ ]:
def evaluate_model(name, y_true, predictions):
    return {
        "Model": name,
        "MAE": mean_absolute_error(y_true, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_true, predictions)),
        "R²": r2_score(y_true, predictions)
    }

results = pd.DataFrame([
    evaluate_model("Linear Regression", y_test, linear_predictions),
    evaluate_model("Random Forest", y_test, rf_predictions)
]).sort_values("R²", ascending=False)

display(results)


## 7. Select the best model and visualize predictions

In [ ]:
best_model_name = results.iloc[0]["Model"]

if best_model_name == "Linear Regression":
    best_model = linear_model
    best_predictions = linear_predictions
else:
    best_model = random_forest_model
    best_predictions = rf_predictions

print("Best model:", best_model_name)


In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(x=y_test, y=best_predictions)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], linestyle="--")
plt.title(f"Actual vs Predicted Prices - {best_model_name}")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.show()


## 8. Predict the price of a new house

In [ ]:
sample_house = pd.DataFrame([{
    "area": 7420,
    "bedrooms": 4,
    "bathrooms": 2,
    "stories": 3,
    "mainroad": "yes",
    "guestroom": "no",
    "basement": "no",
    "hotwaterheating": "no",
    "airconditioning": "yes",
    "parking": 2,
    "prefarea": "yes",
    "furnishingstatus": "furnished"
}])

predicted_price = best_model.predict(sample_house)[0]
print(f"Predicted House Price: {predicted_price:,.2f}")


## 9. Conclusion

The notebook compares Linear Regression and Random Forest regression models using MAE, RMSE, and R². The model with the strongest evaluation results is selected automatically for new-house predictions.

**Next steps:** hyperparameter tuning, cross-validation, feature engineering, or deployment with Streamlit.
